
# FairWarn-SHS — Standard GraphSAGE Baseline

This notebook trains the unmodified GraphSAGE baseline using student node
features and valid peer-study edges.

It uses a two-layer mean-aggregation GraphSAGE network, class-weighted
cross-entropy, early stopping, and five fixed random seeds.


In [ ]:
!pip -q install torch-geometric pandas numpy scikit-learn matplotlib

In [ ]:

from google.colab import files
uploaded = files.upload()

# Upload both:
# FairWarn_SHS_Node_Features.csv
# FairWarn_SHS_Edge_List.csv


In [ ]:

from pathlib import Path
import shutil

Path("src").mkdir(exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/models").mkdir(parents=True, exist_ok=True)
Path("outputs/logs").mkdir(parents=True, exist_ok=True)

shutil.copy(
    "FairWarn_SHS_Node_Features.csv",
    "data/processed/FairWarn_SHS_Node_Features.csv"
)
shutil.copy(
    "FairWarn_SHS_Edge_List.csv",
    "data/processed/FairWarn_SHS_Edge_List.csv"
)


In [ ]:

config_code = '\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nNODE_FILE = ROOT / "data" / "processed" / "FairWarn_SHS_Node_Features.csv"\nEDGE_FILE = ROOT / "data" / "processed" / "FairWarn_SHS_Edge_List.csv"\nOUTPUT_TABLES = ROOT / "outputs" / "tables"\nOUTPUT_FIGURES = ROOT / "outputs" / "figures"\nOUTPUT_MODELS = ROOT / "outputs" / "models"\nOUTPUT_LOGS = ROOT / "outputs" / "logs"\n\nSEEDS = [42, 123, 456, 789, 1010]\n'
Path("src/config.py").write_text(config_code, encoding="utf-8")

script_code = '\nimport argparse\nimport json\nimport random\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport torch\nimport torch.nn.functional as F\n\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.metrics import (\n    roc_auc_score,\n    average_precision_score,\n    precision_score,\n    recall_score,\n    f1_score,\n    balanced_accuracy_score,\n    accuracy_score,\n    brier_score_loss,\n)\nfrom sklearn.model_selection import train_test_split\nfrom torch_geometric.data import Data\nfrom torch_geometric.nn import SAGEConv\n\nfrom config import (\n    NODE_FILE,\n    EDGE_FILE,\n    OUTPUT_TABLES,\n    OUTPUT_FIGURES,\n    OUTPUT_MODELS,\n    OUTPUT_LOGS,\n    SEEDS,\n)\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\ndef load_graph():\n    nodes = pd.read_csv(NODE_FILE)\n    edges = pd.read_csv(EDGE_FILE)\n\n    labelled_mask = (\n        nodes["Label_Available"].eq(1) &\n        nodes["TARGET_AtRisk"].notna()\n    ).to_numpy()\n\n    node_to_index = {\n        node_id: idx for idx, node_id in enumerate(nodes["Node_ID"])\n    }\n\n    source_col = "Source_Node_ID" if "Source_Node_ID" in edges.columns else edges.columns[0]\n    target_col = "Target_Node_ID" if "Target_Node_ID" in edges.columns else edges.columns[1]\n\n    edge_pairs = []\n    for _, row in edges.iterrows():\n        src = row[source_col]\n        dst = row[target_col]\n        if src in node_to_index and dst in node_to_index:\n            s = node_to_index[src]\n            d = node_to_index[dst]\n            edge_pairs.extend([(s, d), (d, s)])\n\n    edge_index = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()\n\n    excluded = {\n        "Node_ID", "Roster_Code", "School_Code", "Class_Code",\n        "Label_Available", "TARGET_AtRisk"\n    }\n    feature_cols = [c for c in nodes.columns if c not in excluded]\n    X_raw = nodes[feature_cols].copy()\n\n    numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()\n    categorical_cols = [c for c in X_raw.columns if c not in numeric_cols]\n\n    for col in numeric_cols:\n        X_raw[col] = X_raw[col].fillna(X_raw[col].median())\n\n    for col in categorical_cols:\n        mode = X_raw[col].mode(dropna=True)\n        X_raw[col] = X_raw[col].fillna(mode.iloc[0] if not mode.empty else "Missing")\n\n    preprocessor = ColumnTransformer([\n        ("numeric", StandardScaler(), numeric_cols),\n        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),\n    ])\n\n    X = preprocessor.fit_transform(X_raw).astype(np.float32)\n    y = nodes["TARGET_AtRisk"].fillna(-1).astype(int).to_numpy()\n\n    data = Data(\n        x=torch.tensor(X, dtype=torch.float32),\n        edge_index=edge_index,\n        y=torch.tensor(y, dtype=torch.long),\n    )\n\n    metadata = {\n        "node_ids": nodes["Node_ID"].tolist(),\n        "labelled_mask": labelled_mask,\n        "nodes": len(nodes),\n        "labelled_nodes": int(labelled_mask.sum()),\n        "unlabelled_nodes": int((~labelled_mask).sum()),\n        "undirected_edges": int(edge_index.shape[1] // 2),\n        "encoded_features": int(X.shape[1]),\n    }\n    return data, metadata\n\ndef make_masks(labels, labelled_mask, seed):\n    labelled_indices = np.where(labelled_mask)[0]\n    labelled_y = labels[labelled_indices]\n\n    train_val_idx, test_idx = train_test_split(\n        labelled_indices,\n        test_size=0.20,\n        stratify=labelled_y,\n        random_state=seed,\n    )\n    train_val_y = labels[train_val_idx]\n    train_idx, val_idx = train_test_split(\n        train_val_idx,\n        test_size=0.1875,\n        stratify=train_val_y,\n        random_state=seed,\n    )\n\n    n = len(labels)\n    train_mask = torch.zeros(n, dtype=torch.bool)\n    val_mask = torch.zeros(n, dtype=torch.bool)\n    test_mask = torch.zeros(n, dtype=torch.bool)\n    train_mask[train_idx] = True\n    val_mask[val_idx] = True\n    test_mask[test_idx] = True\n    return train_mask, val_mask, test_mask\n\nclass GraphSAGE(torch.nn.Module):\n    def __init__(self, in_channels, hidden_channels=64, dropout=0.35):\n        super().__init__()\n        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr="mean")\n        self.conv2 = SAGEConv(hidden_channels, hidden_channels // 2, aggr="mean")\n        self.classifier = torch.nn.Linear(hidden_channels // 2, 2)\n        self.dropout = dropout\n\n    def forward(self, x, edge_index):\n        x = self.conv1(x, edge_index)\n        x = F.relu(x)\n        x = F.dropout(x, p=self.dropout, training=self.training)\n        x = self.conv2(x, edge_index)\n        x = F.relu(x)\n        x = F.dropout(x, p=self.dropout, training=self.training)\n        return self.classifier(x)\n\ndef metrics_from_predictions(y_true, probability, prediction):\n    return {\n        "AUC_ROC": roc_auc_score(y_true, probability),\n        "AUC_PR": average_precision_score(y_true, probability),\n        "Precision_AtRisk": precision_score(y_true, prediction, zero_division=0),\n        "Recall_AtRisk": recall_score(y_true, prediction, zero_division=0),\n        "F1_AtRisk": f1_score(y_true, prediction, zero_division=0),\n        "Weighted_F1": f1_score(y_true, prediction, average="weighted", zero_division=0),\n        "Balanced_Accuracy": balanced_accuracy_score(y_true, prediction),\n        "Accuracy": accuracy_score(y_true, prediction),\n        "Brier_Score": brier_score_loss(y_true, probability),\n    }\n\ndef train_seed(data, metadata, seed, device, max_epochs, patience):\n    set_seed(seed)\n    train_mask, val_mask, test_mask = make_masks(\n        data.y.numpy(), metadata["labelled_mask"], seed\n    )\n\n    graph = data.clone()\n    graph.train_mask = train_mask\n    graph.val_mask = val_mask\n    graph.test_mask = test_mask\n    graph = graph.to(device)\n\n    model = GraphSAGE(graph.num_node_features).to(device)\n\n    train_labels = graph.y[graph.train_mask]\n    class_counts = torch.bincount(train_labels, minlength=2).float()\n    class_weights = class_counts.sum() / (2.0 * class_counts.clamp_min(1.0))\n    class_weights = class_weights.to(device)\n\n    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)\n\n    best_state = None\n    best_val_auc_pr = -np.inf\n    best_epoch = 0\n    wait = 0\n    history = []\n    started = time.perf_counter()\n\n    for epoch in range(1, max_epochs + 1):\n        model.train()\n        optimizer.zero_grad()\n        logits = model(graph.x, graph.edge_index)\n        loss = F.cross_entropy(\n            logits[graph.train_mask],\n            graph.y[graph.train_mask],\n            weight=class_weights,\n        )\n        loss.backward()\n        optimizer.step()\n\n        model.eval()\n        with torch.no_grad():\n            logits = model(graph.x, graph.edge_index)\n            probability = torch.softmax(logits, dim=1)[:, 1]\n            val_true = graph.y[graph.val_mask].cpu().numpy()\n            val_probability = probability[graph.val_mask].cpu().numpy()\n            val_auc_pr = average_precision_score(val_true, val_probability)\n\n        history.append({\n            "Seed": seed,\n            "Epoch": epoch,\n            "Training_Loss": float(loss.item()),\n            "Validation_AUC_PR": float(val_auc_pr),\n        })\n\n        if val_auc_pr > best_val_auc_pr + 1e-6:\n            best_val_auc_pr = val_auc_pr\n            best_epoch = epoch\n            best_state = {\n                key: value.detach().cpu().clone()\n                for key, value in model.state_dict().items()\n            }\n            wait = 0\n        else:\n            wait += 1\n\n        if wait >= patience:\n            break\n\n    training_seconds = time.perf_counter() - started\n\n    model.load_state_dict(best_state)\n    model = model.to(device)\n    model.eval()\n\n    with torch.no_grad():\n        logits = model(graph.x, graph.edge_index)\n        probability = torch.softmax(logits, dim=1)[:, 1]\n        prediction = torch.argmax(logits, dim=1)\n\n    truth = graph.y[graph.test_mask].cpu().numpy()\n    test_probability = probability[graph.test_mask].cpu().numpy()\n    test_prediction = prediction[graph.test_mask].cpu().numpy()\n\n    result = metrics_from_predictions(truth, test_probability, test_prediction)\n    result.update({\n        "Seed": seed,\n        "Best_Epoch": best_epoch,\n        "Validation_AUC_PR": best_val_auc_pr,\n        "Train_Seconds": training_seconds,\n        "Train_N": int(graph.train_mask.sum()),\n        "Validation_N": int(graph.val_mask.sum()),\n        "Test_N": int(graph.test_mask.sum()),\n    })\n\n    test_indices = torch.where(graph.test_mask)[0].cpu().numpy()\n    pred_rows = []\n    for idx, true_value, pred_value, prob in zip(\n        test_indices, truth, test_prediction, test_probability\n    ):\n        pred_rows.append({\n            "Seed": seed,\n            "Node_ID": metadata["node_ids"][idx],\n            "True_Label": int(true_value),\n            "Predicted_Label": int(pred_value),\n            "AtRisk_Probability": float(prob),\n        })\n\n    torch.save({\n        "model_state_dict": best_state,\n        "input_features": graph.num_node_features,\n        "seed": seed,\n        "best_epoch": best_epoch,\n    }, OUTPUT_MODELS / f"graphsage_baseline_seed_{seed}.pt")\n\n    return result, history, pred_rows\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--epochs", type=int, default=500)\n    parser.add_argument("--patience", type=int, default=40)\n    args = parser.parse_args()\n\n    for folder in [OUTPUT_TABLES, OUTPUT_FIGURES, OUTPUT_MODELS, OUTPUT_LOGS]:\n        folder.mkdir(parents=True, exist_ok=True)\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    data, metadata = load_graph()\n\n    print("=" * 70)\n    print("FAIRWARN-SHS STANDARD GRAPHSAGE BASELINE")\n    print("=" * 70)\n    print("Device:", device)\n    print("Nodes:", metadata["nodes"])\n    print("Labelled nodes:", metadata["labelled_nodes"])\n    print("Undirected edges:", metadata["undirected_edges"])\n    print("Encoded features:", metadata["encoded_features"])\n\n    all_metrics, all_history, all_predictions = [], [], []\n\n    for seed in SEEDS:\n        print(f"\\nTraining seed {seed}...")\n        result, history, pred_rows = train_seed(\n            data, metadata, seed, device, args.epochs, args.patience\n        )\n        all_metrics.append(result)\n        all_history.extend(history)\n        all_predictions.extend(pred_rows)\n        print(\n            f"AUC-PR={result[\'AUC_PR\']:.4f} | "\n            f"AUC-ROC={result[\'AUC_ROC\']:.4f} | "\n            f"Recall={result[\'Recall_AtRisk\']:.4f} | "\n            f"F1={result[\'F1_AtRisk\']:.4f} | "\n            f"Best epoch={result[\'Best_Epoch\']}"\n        )\n\n    metrics_df = pd.DataFrame(all_metrics)\n    history_df = pd.DataFrame(all_history)\n    predictions_df = pd.DataFrame(all_predictions)\n\n    metric_cols = [\n        "AUC_ROC", "AUC_PR", "Precision_AtRisk", "Recall_AtRisk",\n        "F1_AtRisk", "Weighted_F1", "Balanced_Accuracy",\n        "Accuracy", "Brier_Score", "Train_Seconds", "Best_Epoch"\n    ]\n\n    summary = {"Model": "Standard GraphSAGE"}\n    for metric in metric_cols:\n        summary[f"{metric}_Mean"] = metrics_df[metric].mean()\n        summary[f"{metric}_SD"] = metrics_df[metric].std(ddof=1)\n\n    summary_df = pd.DataFrame([summary])\n\n    metrics_df.to_csv(OUTPUT_TABLES / "graphsage_seed_metrics.csv", index=False)\n    summary_df.to_csv(OUTPUT_TABLES / "graphsage_summary_mean_sd.csv", index=False)\n    history_df.to_csv(OUTPUT_TABLES / "graphsage_training_history.csv", index=False)\n    predictions_df.to_csv(OUTPUT_TABLES / "graphsage_test_predictions.csv", index=False)\n\n    with open(OUTPUT_LOGS / "graphsage_metadata.json", "w", encoding="utf-8") as f:\n        json.dump({\n            "nodes": metadata["nodes"],\n            "labelled_nodes": metadata["labelled_nodes"],\n            "unlabelled_nodes": metadata["unlabelled_nodes"],\n            "undirected_edges": metadata["undirected_edges"],\n            "encoded_features": metadata["encoded_features"],\n        }, f, indent=2)\n\n    plt.figure(figsize=(8, 5))\n    for seed, group in history_df.groupby("Seed"):\n        plt.plot(group["Epoch"], group["Validation_AUC_PR"], label=f"Seed {seed}")\n    plt.xlabel("Epoch")\n    plt.ylabel("Validation AUC-PR")\n    plt.title("Standard GraphSAGE validation performance")\n    plt.legend()\n    plt.tight_layout()\n    plt.savefig(OUTPUT_FIGURES / "graphsage_validation_auc_pr.png", dpi=300)\n    plt.close()\n\n    print("\\nGRAPHSAGE SUMMARY")\n    print(summary_df.round(4).to_string(index=False))\n\nif __name__ == "__main__":\n    main()\n'
Path("src/03_train_graphsage.py").write_text(script_code, encoding="utf-8")


In [ ]:
!python src/03_train_graphsage.py

In [ ]:

import pandas as pd

summary = pd.read_csv("outputs/tables/graphsage_summary_mean_sd.csv")
seed_results = pd.read_csv("outputs/tables/graphsage_seed_metrics.csv")

display(summary)
display(seed_results)


In [ ]:

from IPython.display import Image, display
display(Image(filename="outputs/figures/graphsage_validation_auc_pr.png"))


In [ ]:

files.download("outputs/tables/graphsage_summary_mean_sd.csv")
files.download("outputs/tables/graphsage_seed_metrics.csv")
files.download("outputs/tables/graphsage_training_history.csv")
files.download("outputs/tables/graphsage_test_predictions.csv")
